In [ ]:
# Parameters
configfile = "config/config.yml"
input_data = "results/data/checkpoints/beforefilter_intermediate_spender_postmortem_labor_klinische_chemie.pq"
targetpop_data = "results/data/checkpoints/targetpop.pq"
display_util = "workflow/scripts/display_util.py"
util = "workflow/scripts/util.py"
output_data = "results/data/intermediate_spender_postmortem_labor_klinische_chemie.pq"
output_model = "results/data/intermediate_spender_postmortem_labor_klinische_chemie.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt
import numpy as np

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
    summarize_index_overlap,
    collist,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    SpenderID,
    drop_duplicate_columns,
    common_translate,
    split_data,
    collapse_col,
    find_redundant_cols,
    fix_redundancies,
    fix_units,
)

### Target Population Filtering

The donors in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
donors = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et", "donor_et_iqtig"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
targetpop = pd.read_parquet(targetpop_data)
data = data[donors.isin(targetpop["donor_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of donors in the data ({donors.nunique()}) and target population ({targetpop["donor_et_id_et"].nunique()})
            to {donors[donors.isin(targetpop["donor_et_id_et"])].nunique()} in the processed data.
        """
    )
)

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

In this file, the {term}`DSO` and {term}`ET` data is sharing one row (see [](general:ic)) if the measurement was taken on the same day. {term}`IQTIG` data is never connected.

In [ ]:
idcols = ["donor_et_dso", "donor_et_id_et", "donor_et_iqtig"]
donordat = {
    col: data.loc[:, [col]].dropna().set_index(col, drop=False) for col in idcols
}
data = split_data(data, idcols)

The following analysis shows the overlap of the donors between the different sources.

In [ ]:
summarize_index_overlap(
    donordat["donor_et_id_et"],
    donordat["donor_et_dso"],
    "ET",
    "DSO",
    c=donordat["donor_et_iqtig"],
    c_label="IQTIG",
)

Even though {term}`IQTIG` data contains some donors, missing the other datasets, it lacks a date column and the specified unit appears to be wrong for some measurements, therefore we drop the {term}`IQTIG` data. The next analysis shows this removed data.

In [ ]:
display_data_doc(data=data["donor_et_iqtig"].reset_index())

In [ ]:
# Drop iqtig, as the values seem unrealistic and have no date asssociated, again they are joined if they have the same data
data = pd.concat(
    (v.reset_index() for k, v in data.items() if k != "donor_et_iqtig")
).reset_index(drop=True)

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

There is no column differentiating between different types of tests (see [](general:rf)). The following analysis compares the data from the different sources. All rows were kept.

In [ ]:
data["Institute with a measurement date"] = (
    (~data["sampling_date_dso"].isna()) + (~data["sampling_date_et"].isna()) * 2
).replace({1: "DSO", 2: "ET", 3: "DSO+ET", 0: "No Date"})
data["donor"] = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
data["sampling_date"] = collapse_col(
    data.loc[:, ["sampling_date_dso", "sampling_date_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
display_long_data_doc(
    data,
    [
        "donor",
    ],
    "sampling_date",
    "Institute with a measurement date",
)
data.drop(
    columns=["sampling_date", "donor", "Institute with a measurement date"],
    inplace=True,
)

### Unit Conversions

First common translations were applied and then we converted different pressure measurements to a common unit (see [](general:uc)). Afterwards, unit specifier columns with only a single unit were removed and measurements with a German numeric formatting and detection boundary detectors converted to numeric.

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

In [ ]:
creatinine_for_comparison = data.loc[
    :, ["creatinine_umol_per_l", "creatinine_unit_et"]
].copy()

fix_units(
    data,
    "alkaline_phosphatase_u_per_l_et",
    "alkaline_phosphatase_unit_et",
    config["data"]["unit_conversions"]["enzymeactivity"]["target"],
    config["data"]["unit_conversions"]["enzymeactivity"]["factors"],
)
fix_units(
    data,
    "amylase_u_per_l_et",
    "amylase_unit_et",
    config["data"]["unit_conversions"]["enzymeactivity"]["target"],
    config["data"]["unit_conversions"]["enzymeactivity"]["factors"],
)
fix_units(
    data,
    "bilirubin_direct_umol_per_l_et",
    "bilirubin_direct_unit_et",
    config["data"]["unit_conversions"]["bilirubin"]["target"],
    config["data"]["unit_conversions"]["bilirubin"]["factors"],
)
fix_units(
    data,
    "bilirubin_all_umol_per_l_et",
    "bilirubin_all_unit_et",
    config["data"]["unit_conversions"]["bilirubin"]["target"],
    config["data"]["unit_conversions"]["bilirubin"]["factors"],
)
fix_units(
    data,
    "calcium_mmol_per_l_et",
    "calcium_unit_et",
    config["data"]["unit_conversions"]["calcium"]["target"],
    config["data"]["unit_conversions"]["calcium"]["factors"],
)
fix_units(
    data,
    "creatine_kinase_u_per_l_et",
    "creatine_kinase_unit_et",
    config["data"]["unit_conversions"]["enzymeactivity"]["target"],
    config["data"]["unit_conversions"]["enzymeactivity"]["factors"],
)
fix_units(
    data,
    "creatine_kinase_mb_unit_et",
    "creatine_kinase_mb_unit_et",
    config["data"]["unit_conversions"]["enzymeactivity"]["target"],
    config["data"]["unit_conversions"]["enzymeactivity"]["factors"],
)
fix_units(
    data,
    "c_reactive_protein_mg_per_l_et",
    "c_reactive_protein_unit_et",
    config["data"]["unit_conversions"]["massconcentration_mg/l"]["target"],
    config["data"]["unit_conversions"]["massconcentration_mg/l"]["factors"],
)
fix_units(
    data,
    "fibrinogen_g_per_l_et",
    "fibrinogen_unit_et",
    config["data"]["unit_conversions"]["massconcentration_g/l"]["target"],
    config["data"]["unit_conversions"]["massconcentration_g/l"]["factors"],
)
fix_units(
    data,
    "gamma_glutamyltransferase_u_per_l_et",
    "gamma_glutamyltransferase_unit_et",
    config["data"]["unit_conversions"]["enzymeactivity"]["target"],
    config["data"]["unit_conversions"]["enzymeactivity"]["factors"],
)
fix_units(
    data,
    "glucose_mmol_per_l_et",
    "glucose_unit_et",
    config["data"]["unit_conversions"]["glucose"]["target"],
    config["data"]["unit_conversions"]["glucose"]["factors"],
)
fix_units(
    data,
    "hemoglobin_g_per_dl_et",
    "hemoglobin_unit_et",
    config["data"]["unit_conversions"]["hemoglobin"]["target"],
    config["data"]["unit_conversions"]["hemoglobin"]["factors"],
)
fix_units(
    data,
    "urea_mmol_per_l_et",
    "urea_unit_et",
    config["data"]["unit_conversions"]["urea"]["target"],
    config["data"]["unit_conversions"]["urea"]["factors"],
)
fix_units(
    data,
    "creatinine_umol_per_l",
    "creatinine_unit_et",
    config["data"]["unit_conversions"]["creatinine_umol/l"]["target"],
    config["data"]["unit_conversions"]["creatinine_umol/l"]["factors"],
)
fix_units(
    data,
    "lactate_dehydrogenase_u_per_l_et",
    "lactate_dehydrogenase_unit_et",
    config["data"]["unit_conversions"]["enzymeactivity"]["target"],
    config["data"]["unit_conversions"]["enzymeactivity"]["factors"],
)
fix_units(
    data,
    "lipase_u_per_l_et",
    "lipase_unit_et",
    config["data"]["unit_conversions"]["enzymeactivity"]["target"],
    config["data"]["unit_conversions"]["enzymeactivity"]["factors"],
)
fix_units(
    data,
    "total_protein_g_per_l_et",
    "total_protein_unit_et",
    config["data"]["unit_conversions"]["massconcentration_g/l"]["target"],
    config["data"]["unit_conversions"]["massconcentration_g/l"]["factors"],
)
fix_units(
    data,
    "glutamic_oxaloacetic_transaminase_u_per_l_et",
    "glutamic_oxaloacetic_transaminase_unit_et",
    config["data"]["unit_conversions"]["enzymeactivity"]["target"],
    config["data"]["unit_conversions"]["enzymeactivity"]["factors"],
)
fix_units(
    data,
    "alanine_transaminase_u_per_l_et",
    "alanine_transaminase_unit_et",
    config["data"]["unit_conversions"]["enzymeactivity"]["target"],
    config["data"]["unit_conversions"]["enzymeactivity"]["factors"],
)
fix_units(
    data,
    "alanine_transaminase_u_per_l_et",
    "alanine_transaminase_unit_et",
    config["data"]["unit_conversions"]["enzymeactivity"]["target"],
    config["data"]["unit_conversions"]["enzymeactivity"]["factors"],
)
fix_units(
    data,
    "troponin_ug_per_l",
    "troponin_unit",
    config["data"]["unit_conversions"]["massconcentration_ug/l"]["target"],
    config["data"]["unit_conversions"]["massconcentration_ug/l"]["factors"],
)
fix_units(
    data,
    "troponin_1_ug_per_l_et",
    "troponin_1_unit_et",
    config["data"]["unit_conversions"]["massconcentration_ug/l"]["target"],
    config["data"]["unit_conversions"]["massconcentration_ug/l"]["factors"],
)
fix_units(
    data,
    "troponin_t_ug_per_l_et",
    "troponin_t_unit_et",
    config["data"]["unit_conversions"]["massconcentration_ug/l"]["target"],
    config["data"]["unit_conversions"]["massconcentration_ug/l"]["factors"],
)

In [ ]:
cols = data.columns[data.columns.to_series().str.contains("_unit")]
assert (data[cols].nunique() != 1).sum() == 0
dropme = cols[data[cols].nunique() == 1]
data = data.drop(columns=dropme)
display(
    Markdown(
        f"The columns {collist(dropme)} were removed as only a single unit was used."
    )
)

In [ ]:
weirdcols = [
    "c_reactive_protein_mg_per_l_dso",
    "troponin_1_ug_per_l_dso",
    "troponin_t_ug_per_l_dso",
]

display(
    Markdown(
        f"""
The columns {collist(weirdcols)} were saved with a `,` as the decimal seperator and contained indicators for measurements, which were outside detection boundaries. These were removed to make consolidation with the other institutes possible."""
    )
)


for col in weirdcols:
    weird = data[col].copy()
    weird = ~weird.isna() & ~weird.str.contains(",", na=False)
    display(
        weird.replace({False: "A numeric value with ,", True: "Detection Indicator"})
        .value_counts()
        .rename("Count of Values")
        .to_frame()
    )
    data.loc[weird, col] = np.nan
    data[col] = pd.to_numeric(
        data[col].str.replace(",", ".", regex=False), errors="raise"
    )

In [ ]:
dropme = ["leukocytes_g_per_l_dso_broken"]
display(
    Markdown(
        f"""
The columns {collist(dropme)} were removed as they also appear to use the wrong units."""
    )
)

data = data.drop(columns=dropme)

We decided to fix `creatinine_umol_per_l_dso_broken`. We will use the other creatinine column for comparison. Based on the density plot of the broken column, we will use a kernel density estimator to impute the broken units.

In [ ]:
ref = creatinine_for_comparison.dropna(how="any").copy()
ref.columns = ["value", "unit"]
ref.groupby("unit").quantile([0, 0.25, 0.5, 0.75, 1])["value"].unstack()

In [ ]:
from sklearn.neighbors import KernelDensity

kdes = ref.groupby("unit")["value"].apply(
    lambda x: KernelDensity(bandwidth="scott", kernel="gaussian").fit(
        x.values.reshape(-1, 1)
    )
)


def get_dens(x):
    if isinstance(x, pd.Series):
        x = x.values
    results = []
    for unit, kde_est in kdes.items():
        miss_mask = np.isnan(x)
        log_dens_not_miss = kde_est.score_samples(x[~miss_mask].reshape(-1, 1))
        log_dens = np.full(x.shape[0], np.nan)
        log_dens[~miss_mask] = log_dens_not_miss
        results.append((unit, log_dens))
    results = pd.DataFrame(dict(results))
    results["x"] = x.flatten()
    best_unit = results.set_index("x").idxmax(axis=1, skipna=True)
    results["best_unit"] = best_unit.values
    return results


result = get_dens(np.asarray([55, 96, 0.62, 1.1]))
result

In [ ]:
broken_fix = get_dens(data["creatinine_umol_per_l_dso_broken"])
broken_fix.dropna(how="any").head(10)

In [ ]:
data["creatinine_unit_dso"] = broken_fix["best_unit"]
broken_before = data["creatinine_umol_per_l_dso_broken"].copy()
fix_units(
    data,
    "creatinine_umol_per_l_dso_broken",
    "creatinine_unit_dso",
    config["data"]["unit_conversions"]["creatinine_umol/l"]["target"],
    config["data"]["unit_conversions"]["creatinine_umol/l"]["factors"],
)

In [ ]:
tocompare = data.loc[
    :, ["creatinine_umol_per_l", "creatinine_umol_per_l_dso_broken"]
].copy()
# boxplot
f, ax = plt.subplots()
tocompare["dso_before_fix"] = broken_before
tocompare.boxplot(
    column=[
        "creatinine_umol_per_l",
        "creatinine_umol_per_l_dso_broken",
        "dso_before_fix",
    ],
    ax=ax,
)
ax.set_yscale("log")
ax.set_ylabel("Creatinine (umol/l)")
ax.set_title("Comparison of ET and DSO Creatinine Measurements")
display(f)

In [ ]:
data.drop(columns=["creatinine_unit_dso"], inplace=True)
data.rename(
    columns={
        "creatinine_umol_per_l_dso_broken": "creatinine_umol_per_l_dso",
        "creatinine_umol_per_l": "creatinine_umol_per_l_et",
    },
    inplace=True,
)

### Consolidating Columns

We consolidated columns that appear for {term}`ET` and {term}`DSO` (see [](general:crc))

In [ ]:
red = find_redundant_cols(data)
# you can manually add if necessary
red["donor_et_id_et"] = ["donor_et_dso", "donor_et_id_et"]
fix_redundancies(data, red)

## Intermediate Dataset

For this longitudinal dataset we recommend the `sampling_date` column as the time axis.

In [ ]:
indcols = ["donor_et_id_et"]
data = data.sort_index(axis=1).sort_values(indcols + ["sampling_date"], axis=0)
data = data.set_index(indcols)

In [ ]:
# Another base class might be necessary, see util.py
# describe columns, without checks for now, order is important
class DonorPostmortemLabBloodGases(SpenderID):
    alanine_transaminase_u_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Alanine Transaminase",
        description="Activity in U/l of the alanine Transaminase",
    )
    albumin_g_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Albumin",
        description="Albumin concentration in g/l",
    )
    alkaline_phosphatase_u_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Alkaline phosphatase",
        description="Activity in U/l of the alkaline phosphatase",
    )
    amylase_u_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Amylase",
        description="Activity in U/l of the amylase",
    )
    antithrombin_3_percent: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Antithrombin 3",
        description="Activity in % of antithrombin 3",
    )
    bilirubin_all_umol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Total Bilirubin",
        description="Concentration of total bilirubin in umol/l",
    )
    bilirubin_direct_umol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Direct Bilirubin",
        description="Concentration of direct bilirubin in umol/l",
    )
    c_reactive_protein_mg_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="C reactive protein",
        description="Concentration of c reactive protein in mg/l",
    )
    calcium_mmol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Calcium",
        description="Concentration of calcium in mmol/l",
    )
    chloride_mmol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Chloride",
        description="Concentration of chloride in mmol/l",
    )
    cholinesterase_u_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Cholinesterase",
        description="Activity in U/l of the cholinesterase",
    )
    communicated_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Communicated date",
        description="Date when the lab result was communicated",
    )
    cholinesterase_u_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Cholinesterase Activity",
        description="Activity in U/l of the cholinesterase",
    )
    creatinine_umol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Creatinine",
        description="Creatinine concentration in umol/l",
    )
    creatine_kinase_mb_u_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Creatine Kinase MB",
        description="Activity in U/l of the creatine kinase MB",
    )
    creatine_kinase_u_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Creatine Kinase",
        description="Activity in U/l of the creatine kinase",
    )
    erythrocytes_t_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Erythrocytes",
        description="Number of Erythrocytes in T/l",
    )
    examination_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Examination date",
        description="Date when the lab measurements were conducted",
    )
    fibrinogen_g_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Fibrinogen",
        description="Fibrinogen concentration in g/l",
    )
    gamma_glutamyltransferase_u_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Gamma Glutamyltransferase",
        description="Activity in U/l of the gamma glutamyltransferase",
    )
    glucose_mmol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Glucose",
        description="Glucose concentration in mmol/l",
    )
    glutamate_dehydrogenase_u_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Glutamate Dehydrogenase",
        description="Activity in U/l of the glutamate dehydrogenase",
    )
    glutamic_oxaloacetic_transaminase_u_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Glutamic oxaloacetic transaminase",
        description="Activity in U/l of the glutamic oxaloacetic transaminase",
    )
    glycated_hemoglobin_percent: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Glycated Hemoglobin",
        description="Fraction of glycated hemoglobin in percent",
    )
    hematocrit_percent: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hematocrit",
        description="Hematocrit value in percent",
    )
    hemoglobin_g_per_dl: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hemoglobin",
        description="Hemoglobin concentration in g/dl",
    )
    international_normalized_ratio: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="INR",
        description="The international normalized ratio",
    )
    lactate_dehydrogenase_u_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Lactate Dehydrogenase",
        description="Activity in U/l of the lactate dehydrogenase",
    )
    lactate_mmol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Lactate",
        description="Lactate concentration in mmol/l",
    )
    leukocytes_g_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Leukocyte",
        description="Leukocyte concentration in g/l",
    )
    lipase_u_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Lipase",
        description="Activity in U/l of the lipase",
    )
    partial_thromboplastin_time_sec: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Partial thromboplastin time",
        description="Partial thromboplastin time in s",
    )
    phosphate_mmol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Phosphate",
        description="Phosphate concentration in mmol/l",
    )
    potassium_mmol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Potassium",
        description="Potassium concentration in mmol/l",
    )
    procalcitonin_ng_per_ml: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Potassium",
        description="Procalcitonin concentration in ng/ml",
    )
    prothrombin_time_sec: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Prothrombin Time",
        description="Prothrombin time in s",
    )
    quick_percent: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Quick",
        description="Quick value in percent",
    )
    result_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Result date",
        description="Date when the lab result was generated",
    )
    sampling_date: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Sampling date",
        description="Date when the sample was taken",
    )
    sodium_mmol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Sodium",
        description="Sodium concentration in mmol/l",
    )
    thrombocytes_g_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Thrombocytes",
        description="Thrombocyte concentration in g/l",
    )
    tissue: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Sampled Tissue",
        description="Tissue the sample was taken from",
        isin=[
            "Blut",
            "Heparinblut",
            "Serum",
            "Liquor",
            "Biopsie",
            "Katheterurin",
            "Bronchiallavage",
            "Drainage",
            "separierte Zellen",
            "Sammelurin",
            "Gewebe",
            "Abstrich",
        ],
    )
    total_protein_g_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Total protein",
        description="Total protein concentration in g/l",
    )
    troponin_1_ug_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Troponin I",
        description="Troponin I concentration in ug/l",
    )
    troponin_t_ug_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Troponin T",
        description="Troponin T concentration in ug/l",
    )
    troponin_ug_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Troponin",
        description="Troponin concentration in ug/l",
    )
    urea_mmol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Urea",
        description="Urea concentration in mmol/l",
    )

    class Config:
        title = "Donor Postmortem Blood Lab Test Dataset"
        description = "Each row represents a blood lab test for a deceased donor. The data is based on the 'element_spender_postmortem_labor_klinische_chemie.csv' file. It contains data from the ET, IQTIG and DSO."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(DonorPostmortemLabBloodGases, data)

In [ ]:
DonorPostmortemLabBloodGases.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    DonorPostmortemLabBloodGases.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)